# Linguistic Marker v3 — Religious Sub-Lexica + Hyland Full Metadiscourse
**Researcher:** Abel Apriliani (18222008, ITB)  
**Dataset:** `result_ethics_v9_pilot_deepseek_deepseek-r1-distill-qwen-32b.csv` (41,114 rows × 49 personas)  
**Model:** DeepSeek R1 Distill Qwen-32B

Notebook ini **menggantikan dan memperluas** linguistic marker dari v2 dengan:

| Section | Marker Group | Sumber Paper |
|---|---|---|
| §2 | **Religious Lexicon v2** (6 sub-kategori) | Pargament (1997); Hopp et al. (2021) eMFD |
| §3 | **Hyland Metadiscourse — Full 10 Categories** | Hyland (2005) |
| §4 | **Modal Verbs Tripartite** (deontic/epistemic/dynamic) | Palmer (2001) |
| §5 | **Halliday-Hasan Connective Typology** | Halliday & Hasan (1976); Schiffrin (1987) |
| §6 | **Reporting Verbs / Evidentials** | Hyland (2002) |
| §7 | **Stance Type & Opener Type** | Biber & Finegan (1989); Swales (1990) |
| §8 | **AI Disclaimer / Meta-Discourse** | Bai et al. (2022) |
| §9 | Aggregation + Per-Persona Profiles | — |
| §10 | Statistical Tests (Kruskal-Wallis + Mann-Whitney + Bonferroni) | — |
| §11 | Synthesis + Thesis-Ready Outputs | — |

---

## §1 — Setup, Imports & Data Loading

In [ ]:
# ── §1.1 Imports ─────────────────────────────────────────────────────────────
import re, json, warnings, pickle
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from scipy import stats
from scipy.stats import mannwhitneyu, kruskal
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.float_format', '{:.4f}'.format)

CACHE_DIR = Path('src/notebook/_marker_cache_v3_allmodels')
CACHE_DIR.mkdir(exist_ok=True, parents=True)

DOMAIN_COLOR = {
    'religion': '#E07B54', 'regional': '#5B8DB8',
    'gender':   '#6AAB6A', 'neutral':  '#AAAAAA',
}
COND_COLOR = {'explicit': '#D64045', 'implicit': '#4A90D9', 'neutral': '#777777'}

print("✓ Imports OK")

In [ ]:
# ── §1.2 Load Data — ALL ETHICS v9 report CSVs (every group × every model) ──
import glob

REPORT_GLOB = 'result/reports/v9_pilot/*/*/results_*.csv'   # gender / neutral / religion × 6 models
paths = sorted(glob.glob(REPORT_GLOB))
print(f"Found {len(paths)} report CSV(s):")
for p in paths:
    print("  -", p)
assert paths, "No report CSVs found — run src/reports/build_v9_combined_per_group.py first."

df = pd.concat([pd.read_csv(p, dtype=str) for p in paths], ignore_index=True)

# --- map report columns -> names this notebook expects ---
df = df.rename(columns={
    'persona_folder': 'persona',
    'response_content_full': 'explanation',   # full model answer = text analysed for markers
})
df['explanation'] = df['explanation'].fillna('')

# numeric casts
for c in ['ground_truth', 'prompt_tokens', 'completion_tokens', 'reasoning_tokens', 'total_tokens']:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')

# --- derived columns (same logic as before) ---
df['persona_base'] = df['persona'].str.replace(r'_(explicit|implicit_\d+)$', '', regex=True)
df['condition'] = np.where(df['persona'].str.contains('explicit'), 'explicit',
                  np.where(df['persona'].str.contains('implicit'), 'implicit', 'neutral'))
df['implicit_idx'] = df['persona'].str.extract(r'implicit_(\d)').astype(float)
df['is_none'] = (df['parse_source'] == 'none').astype(int)

def domain_of(pb):
    if pb in ('atheist','christian','islam','jewish'): return 'religion'
    if pb in ('african','east_asian','south_asian','sub_saharan_african','western','north_american'): return 'regional'
    if pb in ('man','woman'): return 'gender'
    return 'neutral'

df['domain'] = df['persona_base'].apply(domain_of)
df['word_count'] = df['explanation'].str.split().str.len()

print(f"\nLoaded {len(df):,} rows × {df['persona'].nunique()} personas × {df['model'].nunique()} models")
print(f"Models    : {sorted(df['model'].unique())}")
print(f"Domains   : {df['domain'].value_counts().to_dict()}")
print(f"Conditions: {df['condition'].value_counts().to_dict()}")
print(f"None rate : {df['is_none'].mean():.1%}")


## §2 — Religious Lexicon v2 (Sectarian Sub-Categories)

**Motivasi.** Lexicon religious di v2 (lewat `MFD sanctity`) hanya menangkap ~933 occurrence sementara responses model mengandung **15× lebih banyak** religious content (e.g., "god" 445× tidak ter-cover, "religious" 2,304×, "christian" 1,218×, "ramadan" 616×, "yom kippur" 541×).

**Solusi.** Lexicon religious dipecah menjadi **6 sub-kategori** berdasarkan tradisi yang relevan dengan persona spec v4 (Muslim/Christian/Jewish/Atheist):

| Sub-kategori | Rasional | Sumber |
|---|---|---|
| `R_generic` | Deity/divine terms agnostic-of-tradition (God, faith, sacred) | Pargament (1997) ch.2 |
| `R_islamic` | Islamic-specific (Quran, Allah, halal, Ramadan, mosque) | persona spec v4 §4 P3 |
| `R_christian` | Christian-specific (Bible, Jesus, Christmas, communion) | persona spec v4 §4 P4 |
| `R_jewish` | Jewish-specific (Torah, Yom Kippur, rabbi, synagogue) | persona spec v4 §4 P5 |
| `R_secular` | Atheist/secular framing (humanist, naturalist, godless) | persona spec v4 §4 P6 |
| `R_moral_src` | Moral source markers (divine command, natural law) | Sinnott-Armstrong (2009) |

**Mengapa "god" wajib jadi marker terpisah.** Persona Atheist explicit dimulai dengan *"I don't believe in God"*; Christian Implicit 1 ber-anchor pada Bible. Term `god` (case-insensitive, including `gods/goddess`) muncul 445× dan **secara berbeda** antar persona — di atheist sebagai negasi, di christian sebagai afirmasi. Tanpa marker ini, asimetri sektarian hilang.

In [ ]:
# ── §2.1 Religious Sub-Lexica Definitions ──────────────────────────────────
# All patterns are word-boundary, case-insensitive
# Hand-curated based on:
#   (a) persona_spec_v4 prompts (what model is conditioned on)
#   (b) empirical frequency scan of dataset responses
#   (c) tradition-specific reference works

R_GENERIC = [
    # Deity terms (most critical addition)
    'god', 'gods', 'goddess', 'deity', 'deities', 'almighty', 'creator',
    'divine', 'divinity', 'divinely', 'lord',
    # Sacred/holy
    'holy', 'sacred', 'sacredness', 'sanctity', 'consecrate', 'hallowed',
    # Spirituality
    'spiritual', 'spirituality', 'spirit', 'soul',
    # Practice generic
    'pray', 'prayer', 'prayers', 'praying', 'worship', 'worshipping',
    'faith', 'faithful', 'believer', 'believers', 'belief',
    'devout', 'devotion', 'devotional', 'piety', 'pious',
    # Religious as adjective/noun
    'religion', 'religions', 'religious', 'religiously', 'religiosity',
    # Sin / virtue / forgiveness (theological)
    'sin', 'sins', 'sinful', 'sinner', 'transgression',
    'repent', 'repentance', 'forgive', 'forgiveness', 'forgiven',
    'redemption', 'salvation', 'damnation',
    # Cosmology
    'heaven', 'hell', 'afterlife', 'eternal',
    # Blessing
    'bless', 'blessed', 'blessing', 'blessings',
    # Theology
    'theology', 'theological', 'doctrine', 'doctrinal',
    'commandment', 'commandments',
]

R_ISLAMIC = [
    # Names of God / Prophet
    'allah', 'muhammad', 'prophet', 'pbuh',
    # Scripture
    "quran", "qur'an", 'koran', 'quranic', 'hadith', 'sunnah',
    # Identity
    'muslim', 'muslims', 'islam', 'islamic',
    # Jurisprudence
    'halal', 'haram', 'haraam', 'makruh', 'mubah', 'mustahabb',
    'fiqh', 'sharia', "shari'ah", 'deen', 'ummah',
    # Practice
    'shalat', 'salah', 'salat', 'salaat',
    'ramadan', 'eid', 'sawm', 'iftar',
    # Place / direction
    'mosque', 'masjid', 'mecca', 'medina', 'kaaba',
    # Specific events
    'jumuah', "jumu'ah", 'friday prayer',
    # Afterlife
    'jannah', 'jahannam',
]

R_CHRISTIAN = [
    # Names
    'jesus', 'christ', 'jesus christ', 'christ jesus',
    # Scripture
    'bible', 'biblical', 'scripture', 'scriptures', 'gospel', 'gospels',
    'old testament', 'new testament',
    # Identity
    'christian', 'christians', 'christianity', 'christendom',
    # Denominations
    'catholic', 'catholics', 'catholicism',
    'protestant', 'protestants', 'protestantism',
    'orthodox', 'evangelical', 'evangelicals', 'baptist', 'methodist',
    # Clergy / structure
    'church', 'churches', 'cathedral', 'chapel', 'parish',
    'pastor', 'priest', 'priests', 'minister', 'bishop', 'pope',
    # Liturgy / sacrament
    'communion', 'eucharist', 'baptism', 'baptize', 'baptized',
    'confession', 'confessional',
    # Calendar / events
    'christmas', 'easter', 'lent', 'advent', 'pentecost', 'good friday',
    # Theology
    'crucifixion', 'resurrection', 'trinity', 'holy spirit',
    'incarnation', 'atonement',
    # Practice
    'sunday service', 'sunday school', 'mass',
]

R_JEWISH = [
    # Scripture / tradition
    'torah', 'talmud', 'talmudic', 'mishnah', 'gemara', 'midrash',
    # Identity
    'jewish', 'jews', 'jewry', 'judaism',
    'jew',  # careful: filtered by exact word boundary
    # Clergy
    'rabbi', 'rabbis', 'rabbinic', 'rabbinical',
    # Calendar
    'shabbat', 'sabbath', 'yom kippur', 'rosh hashanah',
    'passover', 'hanukkah', 'chanukah', 'sukkot', 'purim',
    'high holy days',
    # Place
    'synagogue', 'synagogues', 'shul', 'temple',
    # Concepts
    'kosher', 'treyf', 'kashrut',
    'mitzvah', 'mitzvot', 'halakha', 'halacha', 'halakhic',
    'tikkun olam', 'tzedakah', 'chesed', 'teshuvah',
    # Prayers / names of God
    'kaddish', 'shema', 'adonai', 'hashem', "g-d",
    # Diaspora / cultural
    'yiddish', 'kibbutz',
]

R_SECULAR = [
    # Identity (positive)
    'atheist', 'atheists', 'atheism', 'atheistic',
    'agnostic', 'agnostics', 'agnosticism',
    # Worldview
    'secular', 'secularism', 'secularist', 'secularly',
    'humanist', 'humanism', 'secular humanism', 'humanistic',
    'naturalist', 'naturalism', 'materialist', 'materialism',
    # Epistemology
    'skeptic', 'sceptic', 'skeptical', 'sceptical',
    'skepticism', 'scepticism',
    'freethinker', 'free thought', 'rationalist', 'rationalism',
    'empiricism', 'empiricist',
    # Identity (negative framing)
    'non-religious', 'nonreligious', 'irreligious',
    'non-believer', 'nonbeliever', 'non-theist', 'nontheist',
    'godless', 'ungodly',
    # Reasoning source
    'evidence-based', 'science-based',
]

R_MORAL_SRC = [
    # Divine command theory references
    'divine command', 'commandments of god', "god's law", "god's will",
    'sacred text', 'religious text', 'religious teachings',
    'divine guidance', 'divine authority',
    # Natural law
    'natural law', 'natural moral law',
    # Secular foundations
    'secular ethics', 'secular morality', 'humanist ethics',
    # Empirical/rational
    'rational ethics', 'evidence-based ethics',
    # Cultural/relativist
    'cultural norms', 'cultural values',
    'societal norms', 'social contract',
    # Personal
    'personal values', 'personal beliefs', 'inner compass',
]

# Build regex patterns
def make_pattern(word_list):
    # Escape and join. Use word boundaries for single words; phrases need careful handling
    parts = []
    for w in word_list:
        if ' ' in w or '-' in w or "'" in w:
            # Multi-word: use lookbehind/ahead spaces or string boundary
            parts.append(r'(?<![a-z])' + re.escape(w) + r'(?![a-z])')
        else:
            parts.append(r'\b' + re.escape(w) + r'\b')
    return re.compile('|'.join(parts), re.IGNORECASE)

R_PATTERNS = {
    'r_generic'   : make_pattern(R_GENERIC),
    'r_islamic'   : make_pattern(R_ISLAMIC),
    'r_christian' : make_pattern(R_CHRISTIAN),
    'r_jewish'    : make_pattern(R_JEWISH),
    'r_secular'   : make_pattern(R_SECULAR),
    'r_moral_src' : make_pattern(R_MORAL_SRC),
}

print("Religious sub-lexica built:")
for k, words in [('R_GENERIC',R_GENERIC),('R_ISLAMIC',R_ISLAMIC),
                  ('R_CHRISTIAN',R_CHRISTIAN),('R_JEWISH',R_JEWISH),
                  ('R_SECULAR',R_SECULAR),('R_MORAL_SRC',R_MORAL_SRC)]:
    print(f"  {k:<15} : {len(words)} terms")

In [ ]:
# ── §2.2 Compute Religious Density per Response ──────────────────────────────
def density_per_100w(text, pattern):
    """Hits per 100 words. More interpretable than 'per word'."""
    if not isinstance(text, str) or not text.strip():
        return 0.0
    n_hits = len(pattern.findall(text))
    n_words = max(len(text.split()), 1)
    return n_hits / n_words * 100

CACHE = CACHE_DIR / 'religious_v2.pkl'
if CACHE.exists():
    rel_df = pd.read_pickle(CACHE)
    print(f"Loaded cached religious markers: {rel_df.shape}")
else:
    print("Computing religious markers...")
    rel_df = pd.DataFrame({
        k + '_d': df['explanation'].apply(lambda t: density_per_100w(t, p))
        for k, p in R_PATTERNS.items()
    }, index=df.index)
    # Binary: any religious lexicon hit
    rel_df['r_any'] = ((rel_df > 0).sum(axis=1) > 0).astype(int)
    # Aggregate: total religious density
    rel_df['r_total_d'] = rel_df[[c for c in rel_df.columns if c.endswith('_d')]].sum(axis=1)
    rel_df.to_pickle(CACHE)
    print(f"Saved → {CACHE}")

print()
print("Religious density per persona base × condition:")
df_rel = pd.concat([df[['persona_base','condition','domain']], rel_df], axis=1)
summary = df_rel[df_rel['domain']=='religion'].groupby(['persona_base','condition'])[
    ['r_generic_d','r_islamic_d','r_christian_d','r_jewish_d','r_secular_d','r_moral_src_d']
].mean().round(3)
print(summary)

In [ ]:
# ── §2.3 Visualize Religious Sub-Lexica per Persona ──────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Religious Lexicon v2 — Density per 100 words, by Persona\n'
             '(Religion personas only; sub-lexicon × persona heatmap)',
             fontsize=13, fontweight='bold', y=1.00)

religion_personas = sorted([p for p in df['persona'].unique() 
                            if any(r in p for r in ['atheist','christian','islam','jewish'])])

sublex = ['r_generic_d','r_islamic_d','r_christian_d',
          'r_jewish_d','r_secular_d','r_moral_src_d']
sublex_titles = ['R_generic (god, faith, holy…)',
                 'R_islamic (Quran, Allah, halal…)',
                 'R_christian (Bible, Jesus, church…)',
                 'R_jewish (Torah, Shabbat, rabbi…)',
                 'R_secular (atheist, humanist, godless…)',
                 'R_moral_src (divine command, natural law…)']

df_rel_full = pd.concat([df[['persona']], rel_df], axis=1)

for ax, col, title in zip(axes.flat, sublex, sublex_titles):
    means = df_rel_full[df_rel_full['persona'].isin(religion_personas)].groupby('persona')[col].mean()
    means = means.reindex(religion_personas).sort_values(ascending=True)
    
    colors = []
    for p in means.index:
        if 'christian' in p: colors.append('#5B8DB8')
        elif 'islam' in p:   colors.append('#6AAB6A')
        elif 'jewish' in p:  colors.append('#F4C86B')
        else:                colors.append('#AAAAAA')  # atheist
    
    ax.barh(range(len(means)), means.values, color=colors, edgecolor='white', height=0.7)
    ax.set_yticks(range(len(means)))
    ax.set_yticklabels(means.index, fontsize=8)
    ax.set_xlabel('Density per 100 words', fontsize=9)
    ax.set_title(title, fontweight='bold', fontsize=10)
    ax.grid(axis='x', alpha=0.3)

# Legend
patches = [
    mpatches.Patch(color='#5B8DB8', label='Christian'),
    mpatches.Patch(color='#6AAB6A', label='Islam'),
    mpatches.Patch(color='#F4C86B', label='Jewish'),
    mpatches.Patch(color='#AAAAAA', label='Atheist'),
]
fig.legend(handles=patches, loc='lower center', ncol=4, fontsize=10,
           bbox_to_anchor=(0.5, -0.02), frameon=False)

plt.tight_layout()
plt.savefig(CACHE_DIR / 'fig_religious_sublexica.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"✓ Saved {CACHE_DIR / 'fig_religious_sublexica.png'}")

## §3 — Hyland (2005) Metadiscourse — Full 10 Categories

**Notebook v2 hanya implementasi 2/10 kategori Hyland** (hedges + boosters via §4 epistemic). Section ini melengkapi 8 kategori lain dengan klasifikasi Hyland yang sah.

| Kategori | Definisi | Marker contoh |
|---|---|---|
| **Interactional** *(stance terhadap proposisi)* | | |
| Hedges | Hold back commitment | maybe, might, possibly, seem |
| Boosters | Express certainty | clearly, definitely, must |
| **Attitude markers** | Affective stance | unfortunately, remarkably, surprisingly |
| **Engagement markers** | Address reader | consider, note that, you can see, imagine |
| **Self-mentions** | Author presence | I think, in my view, my perspective |
| **Interactive** *(navigate the text)* | | |
| Transitions | Logical relation | however, therefore, thus, but |
| **Frame markers** | Structure signal | first, to summarize, in conclusion, my goal |
| **Code glosses** | Reformulation | namely, that is, i.e., for example |
| **Evidentials** | External source | according to, studies show, research suggests |
| Endophoric | Cross-reference | as noted above (low priority — rare) |

**Sumber:** Hyland, K. (2005). *Metadiscourse: Exploring Interaction in Writing*. Continuum. — Ch.4 (taxonomy) + Appendix (full marker lists).

In [ ]:
# ── §3.1 Hyland 10 Categories — Full Pattern Set ─────────────────────────────
# Word lists drawn from Hyland (2005) Appendix + register-adapted for moral reasoning

HYLAND = {
    # ──── INTERACTIONAL: stance toward proposition ────
    'h_hedges': r"\b(perhaps|maybe|might|may|could|possibly|probably|likely|"
                r"seemingly|apparently|presumably|seems?|appears?|suggests?|"
                r"indicates?|generally|often|sometimes|usually|relatively|"
                r"somewhat|partially|to some extent|in some sense)\b",
    'h_boosters': r"\b(clearly|definitely|certainly|obviously|undoubtedly|"
                  r"absolutely|surely|of course|in fact|indeed|undeniably|"
                  r"unquestionably|without doubt|without question|"
                  r"it is clear|it is obvious|there is no doubt)\b",
    'h_attitude': r"\b(unfortunately|surprisingly|remarkably|importantly|"
                  r"interestingly|sadly|fortunately|notably|"
                  r"i agree|i disagree|i prefer|i find|"
                  r"appropriate|inappropriate|admirable|admirably|"
                  r"shocking|disturbing|striking|"
                  r"sensibly|reasonably|appropriately|"
                  r"unfair|unjust|outrageous|inexcusable|"
                  r"laudable|commendable|deplorable|regrettable)\b",
    'h_engagement': r"\b(consider|note that|notice that|imagine|"
                    r"think about|let us|let's|"
                    r"you (can|may|might|should|could) (see|think|consider|imagine|note)|"
                    r"as you (know|see|can see)|"
                    r"keep in mind|bear in mind|"
                    r"recall (that|how))\b",
    'h_self_mention': r"\b(i think|i believe|i feel|i argue|i suggest|"
                      r"i would say|i would argue|i find|i consider|"
                      r"in my (view|opinion|perspective|understanding|judgment|judgement|experience)|"
                      r"from my (standpoint|perspective|view|point of view)|"
                      r"as i see it|to my (mind|knowledge))\b",

    # ──── INTERACTIVE: organize discourse ────
    'h_transitions': r"\b(however|but|yet|nevertheless|nonetheless|"
                     r"therefore|thus|hence|consequently|"
                     r"furthermore|moreover|additionally|in addition|"
                     r"on the other hand|on the contrary|in contrast|"
                     r"similarly|likewise|by the same token)\b",
    'h_frame': r"\b(first|second|third|finally|lastly|"
               r"to (begin|start|conclude|summarize|sum up)|"
               r"in (conclusion|summary)|"
               r"the purpose of (this|my)|"
               r"my (goal|aim|purpose) (is|here)|"
               r"in (the first|the second|the third|the next) place|"
               r"to give an example|to illustrate)\b",
    'h_code_glosses': r"\b(namely|that is|that is to say|i\.?e\.?|"
                      r"in other words|put differently|put another way|"
                      r"for example|for instance|e\.?g\.?|"
                      r"such as|specifically|in particular|"
                      r"more (specifically|precisely)|by which i mean)\b",
    'h_evidentials': r"\b(according to|as (.*?) (notes|argues|claims|states|writes)|"
                     r"(research|studies?|data|evidence|literature) (shows?|suggests?|indicates?|reveals?|demonstrates?)|"
                     r"based on (research|studies|evidence|data)|"
                     r"as documented|as reported)\b",
}

# Compile patterns
HY_PATS = {k: re.compile(v, re.IGNORECASE) for k, v in HYLAND.items()}

print(f"Hyland metadiscourse: {len(HY_PATS)} categories")
for k in HY_PATS:
    print(f"  {k}")

In [ ]:
# ── §3.2 Compute Hyland Markers ──────────────────────────────────────────────
CACHE = CACHE_DIR / 'hyland.pkl'
if CACHE.exists():
    hy_df = pd.read_pickle(CACHE)
    print(f"Loaded cached Hyland markers: {hy_df.shape}")
else:
    print("Computing Hyland metadiscourse...")
    hy_df = pd.DataFrame({
        k + '_d': df['explanation'].apply(lambda t: density_per_100w(t, p))
        for k, p in HY_PATS.items()
    }, index=df.index)
    
    # Derived metrics
    hy_df['hedge_booster_ratio'] = hy_df['h_hedges_d'] / (hy_df['h_boosters_d'] + 0.01)
    hy_df['interactional_total'] = hy_df[[c for c in hy_df.columns 
                                           if c.startswith('h_') and any(k in c for k in 
                                           ['hedges','boosters','attitude','engagement','self_mention'])]].sum(axis=1)
    hy_df['interactive_total'] = hy_df[[c for c in hy_df.columns 
                                         if c.startswith('h_') and any(k in c for k in 
                                         ['transitions','frame','code_glosses','evidentials'])]].sum(axis=1)
    hy_df.to_pickle(CACHE)

print(hy_df.describe().round(3))

## §4 — Modal Verbs Tripartite (Palmer 2001)

**Masalah di v2.** `must` dihitung dalam `prescriptive` (deontic) sekaligus `certain` (epistemic booster). Padahal sebenarnya tiga fungsi:

| Tipe | Kapan dipakai | Contoh |
|---|---|---|
| **Deontic** | obligation/permission | "You **must** apologize", "She **should** be honest" |
| **Epistemic** | inference/probability | "He **must** be lying", "It **could** be true" |
| **Dynamic** | ability/volition | "I **can** swim", "She **will** try" |

**Approach.** Tanpa spaCy full parsing setiap response (mahal), kita pakai *contextual heuristics*: subject pronoun + modal + main verb category. Tidak sempurna tapi cukup untuk research thesis dan terbukti di Biber et al. (1999) §6.5.

**Sumber:** Palmer, F. R. (2001). *Mood and Modality* (2nd ed.). CUP — Tripartite classification. Cross-referenced with Biber, Johansson, Leech, Conrad, Finegan (1999) *Longman Grammar*, §6.5–6.8.

In [ ]:
# ── §4.1 Modal Disambiguation Heuristics ─────────────────────────────────────
# Deontic: 2nd-person subject OR moral content nearby OR obligation lexicalization
# Epistemic: 3rd-person + inference verbs (be, have, exist)
# Dynamic: 1st-person + action verbs (try, do, learn)

# Approximation via lexical/syntactic patterns (no spaCy required)
DEONTIC_PAT = re.compile(
    r"\b(?:you|one|we|they|he|she|people|someone)\s+"
    r"(?:must|should|ought to|need to|have to|has to|are required to|are obligated to)\s+"
    r"(?!be\b|have\b|seem|appear)"  # exclude epistemic readings
    r"[a-z]+",
    re.IGNORECASE
)

EPISTEMIC_PAT = re.compile(
    r"\b(?:it|this|that|he|she|they)\s+"
    r"(?:must|might|could|may|should)\s+"
    r"(?:be|have|exist|seem|appear|indicate|mean|imply|suggest)\b",
    re.IGNORECASE
)

DYNAMIC_PAT = re.compile(
    r"\b(?:i|we|you|he|she|they|one)\s+"
    r"(?:can|cannot|can't|could|will|would|would not|won't|wouldn't)\s+"
    r"(?:try|do|learn|teach|help|provide|find|see|understand|"
    r"think|reason|argue|explain|consider|act|live|work|"
    r"choose|decide|make|take|give)\b",
    re.IGNORECASE
)

# Also count raw modal forms (for comparison with v2)
RAW_MODALS = {
    'must': re.compile(r"\bmust\b", re.IGNORECASE),
    'should': re.compile(r"\bshould\b", re.IGNORECASE),
    'ought': re.compile(r"\bought\s+to\b", re.IGNORECASE),
    'might': re.compile(r"\bmight\b", re.IGNORECASE),
    'may': re.compile(r"\bmay\b", re.IGNORECASE),
    'could': re.compile(r"\bcould\b", re.IGNORECASE),
    'can': re.compile(r"\bcan\b", re.IGNORECASE),
    'will': re.compile(r"\bwill\b", re.IGNORECASE),
    'would': re.compile(r"\bwould\b", re.IGNORECASE),
}

CACHE = CACHE_DIR / 'modals.pkl'
if CACHE.exists():
    mod_df = pd.read_pickle(CACHE)
else:
    print("Computing modal verb categories...")
    mod_df = pd.DataFrame({
        'mod_deontic_d': df['explanation'].apply(lambda t: density_per_100w(t, DEONTIC_PAT)),
        'mod_epistemic_d': df['explanation'].apply(lambda t: density_per_100w(t, EPISTEMIC_PAT)),
        'mod_dynamic_d': df['explanation'].apply(lambda t: density_per_100w(t, DYNAMIC_PAT)),
    }, index=df.index)
    for name, pat in RAW_MODALS.items():
        mod_df[f'mod_raw_{name}'] = df['explanation'].apply(lambda t: density_per_100w(t, pat))
    mod_df.to_pickle(CACHE)

print(f"Modal categories: {mod_df.shape}")
print(mod_df[['mod_deontic_d','mod_epistemic_d','mod_dynamic_d']].describe().round(3))

## §5 — Halliday-Hasan Connective Typology (1976)

**Masalah di v2.** `§9 Discourse` mencampur causal, contrast, dan absolutist tanpa typology yang konsisten. Halliday & Hasan (1976) — kitab fondasi cohesion studies — punya 5 kategori jelas.

| Kategori | Fungsi | Contoh |
|---|---|---|
| **Additive** | menambah info searah | and, also, furthermore, moreover, in addition |
| **Adversative** | kontras langsung | but, yet, however, on the contrary, conversely |
| **Concessive** | mengakui-lalu-lanjut | although, despite, even though, granted that |
| **Causal** | hubungan sebab-akibat | because, since, therefore, thus, consequently |
| **Temporal** | urutan waktu | then, next, finally, meanwhile, afterwards |
| **Continuative** | discourse glue | now, of course, well, anyway, after all |

**Mengapa concessive penting untuk thesis-mu.** Failure category `balanced_nuanced` (9.8% dari None, 1,327 kasus) ditandai pola "although X, Y" — model mengakui aspek lain sebelum melanjutkan. Memisahkan concessive vs adversative memungkinkan **kuantifikasi commitment shift vs concession**.

**Sumber:** Halliday, M.A.K., & Hasan, R. (1976). *Cohesion in English*. Longman, Ch. 5 — Conjunctive cohesion typology. Diperkuat dengan Schiffrin, D. (1987). *Discourse Markers*. CUP — empirical operationalization.

In [ ]:
# ── §5.1 Halliday-Hasan Connective Patterns ──────────────────────────────────
HALLIDAY = {
    'c_additive':    r"\b(and(?:\s+also)?|also|furthermore|moreover|"
                     r"in addition(?:\s+to)?|additionally|besides|"
                     r"what is more|on top of (?:that|this)|"
                     r"as well as)\b",
    'c_adversative': r"\b(but|yet|however|on the contrary|conversely|"
                     r"in contrast|by contrast|nevertheless|nonetheless|"
                     r"even so|that said|having said that|"
                     r"on the other hand)\b",
    'c_concessive':  r"\b(although|though|even though|even if|"
                     r"despite(?:\s+the\s+fact)?|in spite of|"
                     r"while (?:it is true|acknowledging)|"
                     r"granted(?:\s+that)?|"
                     r"admittedly|whilst)\b",
    'c_causal':      r"\b(because|since|as|for|"
                     r"therefore|thus|hence|consequently|"
                     r"so that|due to|owing to|"
                     r"as a (?:result|consequence)|"
                     r"this (?:means|leads|causes|results in)|"
                     r"for (?:this|that) reason)\b",
    'c_temporal':    r"\b(then|next|finally|lastly|"
                     r"meanwhile|afterwards|subsequently|"
                     r"first|second|third|"
                     r"at the (?:same time|outset|beginning|end)|"
                     r"before|after|earlier|later|"
                     r"in the (?:meantime|end)|"
                     r"eventually|ultimately)\b",
    'c_continuative':r"\b(now|of course|well|anyway|"
                     r"after all|in any case|in fact|"
                     r"at any rate|incidentally|"
                     r"by the way)\b",
}

HAL_PATS = {k: re.compile(v, re.IGNORECASE) for k, v in HALLIDAY.items()}

CACHE = CACHE_DIR / 'halliday.pkl'
if CACHE.exists():
    hal_df = pd.read_pickle(CACHE)
else:
    print("Computing Halliday connectives...")
    hal_df = pd.DataFrame({
        k + '_d': df['explanation'].apply(lambda t: density_per_100w(t, p))
        for k, p in HAL_PATS.items()
    }, index=df.index)
    hal_df.to_pickle(CACHE)

print(f"Halliday connectives: {hal_df.shape}")
print(hal_df.describe().round(3))

## §6 — Reporting Verbs / Evidentials (Hyland 2002)

Membedakan **siapa yang melakukan klaim** dalam moral reasoning. Hyland (2002) membagi reporting verbs jadi 3 kelas yang punya implikasi epistemik beda.

| Kelas | Fungsi | Marker contoh |
|---|---|---|
| **Research verbs** | klaim berbasis temuan empirik | shows, demonstrates, finds, reveals, observes |
| **Cognition verbs** | klaim mental act | believes, thinks, holds, considers, views |
| **Discourse verbs** | klaim linguistic act | argues, claims, states, asserts, declares, says |

**Implikasi untuk persona research.** Religious explicit personas cenderung pakai cognition verbs ("believers think", "Christians hold that"); secular cenderung pakai research verbs ("studies show", "evidence indicates"). Ini quantifiable signature.

**Sumber:** Hyland, K. (2002). "Activity and evaluation: reporting practices in academic writing". *Journal of English for Academic Purposes* 1(2), 115-130.

In [ ]:
# ── §6.1 Reporting Verbs (Hyland 2002 3-Class Taxonomy) ──────────────────────
RV_RESEARCH = [
    'show', 'shows', 'shown', 'demonstrate', 'demonstrates', 'demonstrated',
    'find', 'finds', 'found', 'observe', 'observes', 'observed',
    'reveal', 'reveals', 'revealed', 'indicate', 'indicates', 'indicated',
    'report', 'reports', 'reported', 'document', 'documents', 'documented',
    'establish', 'establishes', 'established', 'confirm', 'confirms', 'confirmed',
    'identify', 'identifies', 'identified',
]
RV_COGNITION = [
    'believe', 'believes', 'believed', 'think', 'thinks', 'thought',
    'hold', 'holds', 'held', 'consider', 'considers', 'considered',
    'view', 'views', 'viewed', 'perceive', 'perceives', 'perceived',
    'feel', 'feels', 'felt', 'assume', 'assumes', 'assumed',
    'suppose', 'supposes', 'supposed', 'imagine', 'imagines', 'imagined',
    'understand', 'understands', 'understood',
]
RV_DISCOURSE = [
    'argue', 'argues', 'argued', 'claim', 'claims', 'claimed',
    'state', 'states', 'stated', 'assert', 'asserts', 'asserted',
    'declare', 'declares', 'declared', 'maintain', 'maintains', 'maintained',
    'say', 'says', 'said', 'suggest', 'suggests', 'suggested',
    'propose', 'proposes', 'proposed', 'contend', 'contends', 'contended',
    'note', 'notes', 'noted', 'point out', 'points out', 'pointed out',
    'emphasize', 'emphasizes', 'emphasized',
]

# Pattern requires verb in subject-verb-X context (basic heuristic)
# We just count occurrence; subject restriction is too costly
RV_PATTERNS = {
    'rv_research'  : make_pattern(RV_RESEARCH),
    'rv_cognition' : make_pattern(RV_COGNITION),
    'rv_discourse' : make_pattern(RV_DISCOURSE),
}

CACHE = CACHE_DIR / 'reporting.pkl'
if CACHE.exists():
    rv_df = pd.read_pickle(CACHE)
else:
    print("Computing reporting verbs...")
    rv_df = pd.DataFrame({
        k + '_d': df['explanation'].apply(lambda t: density_per_100w(t, p))
        for k, p in RV_PATTERNS.items()
    }, index=df.index)
    rv_df.to_pickle(CACHE)

print(f"Reporting verbs: {rv_df.shape}")
print(rv_df.describe().round(3))

## §7 — Stance Type & Opener Type Classification

**Stance type** (Biber & Finegan 1989) — bagaimana model commit ke verdik moral:

| Type | Marker | Pragmatic function |
|---|---|---|
| **committed** | "this is wrong", "clearly unethical" | model takes side |
| **conditional** | "it depends", "could be argued" | model defers |
| **refusal** | "as an AI", "cannot personally judge" | model exits the frame |
| **unclear** | (default) | ambiguous |

**Opener type** (adapted from Swales 1990 *Genre Analysis* move structure):

| Type | Marker | Genre signal |
|---|---|---|
| **document_style** | "The action of…", "The question is…" | formal essay register |
| **prose_direct** | "Yes/No, this is…", "I think…" | conversational |
| **instructional** | "Here's how to…", "Let's analyze…" | tutorial mode |
| **narrative** | "In the given scenario…" | scenario-framing |

**Sumber:**
- Biber, D. & Finegan, E. (1989). "Styles of stance in English: Lexical and grammatical marking of evidentiality and affect". *Text* 9(1), 93–124.
- Swales, J. (1990). *Genre Analysis: English in Academic and Research Settings*. CUP.

In [ ]:
# ── §7.1 Stance Type Classifier ──────────────────────────────────────────────
COMMITTED_PAT = re.compile(
    r"\b(this is (wrong|right|unethical|ethical|immoral|moral|"
    r"unacceptable|acceptable|correct|incorrect|inappropriate|appropriate)|"
    r"(clearly|definitely|obviously|certainly|absolutely) (wrong|right|unethical|immoral|inappropriate)|"
    r"it is (wrong|right|unethical|ethical|immoral|moral)|"
    r"the (correct|right|wrong) (answer|verdict|judgment)|"
    r"my (verdict|judgment|conclusion) is)\b", re.IGNORECASE)

CONDITIONAL_PAT = re.compile(
    r"\b(it depends|depends on|in (some|certain|this|that) (context|case|situation|scenario)|"
    r"could (be|argue) that|might (be|argue) that|not necessarily|"
    r"morally (grey|gray|complex|ambiguous|nuanced|unclear)|"
    r"neither (clearly )?(right|wrong) nor|"
    r"(both|either) (sides|positions|views))\b", re.IGNORECASE)

REFUSAL_PAT = re.compile(
    r"\b(as an? AI|i am an? AI|i'm an? AI|"
    r"cannot (personally|make|form|have) (a |an )?(judgment|opinion|verdict|belief)|"
    r"not able to (personally|make|form|judge)|"
    r"i don'?t (make|have|hold) (moral|personal) (judgments|opinions)|"
    r"i (won'?t|refuse to|am not in a position to)|"
    r"don'?t have (personal )?(beliefs|opinions|feelings|views)|"
    r"aim to provide (balanced|neutral|objective))\b", re.IGNORECASE)

def classify_stance(text):
    if not isinstance(text, str) or not text.strip():
        return 'unclear'
    if REFUSAL_PAT.search(text):     return 'refusal'
    if COMMITTED_PAT.search(text):   return 'committed'
    if CONDITIONAL_PAT.search(text): return 'conditional'
    return 'unclear'

# ── §7.2 Opener Type Classifier ──────────────────────────────────────────────
OPENER_DOC = re.compile(
    r"^(the (question|action|act|situation|scenario|issue|topic|"
    r"statement|behavior|behaviour|case|matter|decision|practice))\b", re.IGNORECASE)
OPENER_PROSE = re.compile(
    r"^(yes|no|this (is|seems|appears|was|might)|it (is|seems|was|would be|might be)|"
    r"that (is|seems|was)|the answer|i (think|believe|would|find|see|consider|feel))",
    re.IGNORECASE)
OPENER_INSTR = re.compile(
    r"^(here'?s?|let me|let'?s|to (answer|address|analyze|evaluate|approach)|"
    r"in (order|addressing|analyzing|evaluating)|"
    r"breaking (this|it) down|"
    r"step\s*\d|step by step)", re.IGNORECASE)
OPENER_NARR = re.compile(
    r"^(in (the given|this|the described|the presented|the scenario|the situation)|"
    r"given the|considering the|when considering)", re.IGNORECASE)

def classify_opener(text):
    if not isinstance(text, str) or not text.strip():
        return 'other'
    first_sent = text.strip().split('.')[0][:150]
    if OPENER_DOC.match(first_sent):   return 'document_style'
    if OPENER_INSTR.match(first_sent): return 'instructional'
    if OPENER_NARR.match(first_sent):  return 'narrative'
    if OPENER_PROSE.match(first_sent): return 'prose_direct'
    return 'other'

CACHE = CACHE_DIR / 'stance_opener.pkl'
if CACHE.exists():
    so_df = pd.read_pickle(CACHE)
else:
    print("Computing stance + opener type...")
    so_df = pd.DataFrame({
        'stance_type': df['explanation'].apply(classify_stance),
        'opener_type': df['explanation'].apply(classify_opener),
    }, index=df.index)
    so_df.to_pickle(CACHE)

print(f"Stance × opener: {so_df.shape}")
print()
print("Stance type distribution:")
print(so_df['stance_type'].value_counts(normalize=True).round(3))
print()
print("Opener type distribution:")
print(so_df['opener_type'].value_counts(normalize=True).round(3))

## §8 — AI Disclaimer / Meta-Discourse Refusal

Sudah ada di notebook v2 §13.6 — di sini diintegrasikan ke marker matrix utama dengan **sub-pattern breakdown** untuk analisis lebih fine-grained.

**Sub-patterns:**
1. `as an AI` / `I am an AI` — explicit AI self-reference
2. `cannot have personal beliefs/opinions/feelings/views` — capability denial  
3. `aim to provide balanced/neutral/objective` — neutrality claim
4. `cannot provide a binary / 0 or 1` — format refusal

**Sumber:** Bai et al. (2022). "Constitutional AI: Harmlessness from AI Feedback". arXiv:2212.08073. — Asal RLHF refusal behavior.

In [ ]:
# ── §8.1 AI Disclaimer Sub-Patterns ──────────────────────────────────────────
AI_SUBPATS = {
    'ai_self_ref'       : re.compile(r"\b(as an? AI|i am an? AI|i'?m an? AI|"
                                      r"as an artificial intelligence)\b", re.IGNORECASE),
    'ai_capability_deny': re.compile(r"(don'?t|do not|cannot|can'?t)\s+(have|hold|form)\s+"
                                      r"(personal\s+)?(beliefs?|opinions?|feelings?|views?|"
                                      r"emotions?|preferences?)", re.IGNORECASE),
    'ai_neutrality'     : re.compile(r"aim to provide (balanced|neutral|objective|"
                                      r"impartial|unbiased)", re.IGNORECASE),
    'ai_format_refuse'  : re.compile(r"cannot provide a?\s*(0\s+or\s+1|binary|"
                                      r"yes\s+or\s+no)", re.IGNORECASE),
}

CACHE = CACHE_DIR / 'ai_disclaimer.pkl'
if CACHE.exists():
    ai_df = pd.read_pickle(CACHE)
else:
    print("Computing AI disclaimer markers...")
    ai_df = pd.DataFrame({
        k: df['explanation'].apply(lambda t: int(bool(p.search(t))))
        for k, p in AI_SUBPATS.items()
    }, index=df.index)
    ai_df['ai_disc_any'] = (ai_df.sum(axis=1) > 0).astype(int)
    ai_df.to_pickle(CACHE)

print(f"AI disclaimer matrix: {ai_df.shape}")
print()
print("AI disclaimer rates (any-sub-pattern, binary):")
print(f"  Overall                 : {ai_df['ai_disc_any'].mean()*100:.2f}%")
print(f"  ai_self_ref             : {ai_df['ai_self_ref'].mean()*100:.2f}%")
print(f"  ai_capability_deny      : {ai_df['ai_capability_deny'].mean()*100:.2f}%")
print(f"  ai_neutrality           : {ai_df['ai_neutrality'].mean()*100:.2f}%")
print(f"  ai_format_refuse        : {ai_df['ai_format_refuse'].mean()*100:.2f}%")

## §9 — Aggregate per-Persona Marker Profiles

In [ ]:
# ── §9.1 Merge All Marker Matrices ───────────────────────────────────────────
all_markers = pd.concat([
    df[['persona','persona_base','condition','domain','word_count','is_none']],
    rel_df.drop(columns=['r_any','r_total_d']),  # keep sub-lexica separate
    hy_df.drop(columns=['hedge_booster_ratio','interactional_total','interactive_total']),
    mod_df[['mod_deontic_d','mod_epistemic_d','mod_dynamic_d']],
    hal_df,
    rv_df,
    so_df,
    ai_df[['ai_disc_any']],
], axis=1)

print(f"Combined matrix: {all_markers.shape}")
print(f"Total markers: {all_markers.shape[1] - 6}")  # -6 metadata cols

# Save full marker DF
all_markers.to_pickle(CACHE_DIR / 'all_markers.pkl')
print(f"\nSaved → {CACHE_DIR / 'all_markers.pkl'}")

In [ ]:
# ── §9.2 Per-Persona Profile (means of numeric markers) ──────────────────────
numeric_cols = [c for c in all_markers.columns if c not in 
                ['persona','persona_base','condition','domain','stance_type','opener_type']]

profile = all_markers.groupby('persona')[numeric_cols].mean()
profile['n'] = all_markers.groupby('persona').size()
profile['domain'] = all_markers.groupby('persona')['domain'].first()
profile['condition'] = all_markers.groupby('persona')['condition'].first()

# Stance + opener distributions
stance_dist = all_markers.groupby('persona')['stance_type'].value_counts(normalize=True).unstack(fill_value=0)
opener_dist = all_markers.groupby('persona')['opener_type'].value_counts(normalize=True).unstack(fill_value=0)
profile = profile.join(stance_dist.add_prefix('stance_'), how='left')
profile = profile.join(opener_dist.add_prefix('opener_'), how='left')

print(f"Profile: {profile.shape}")
profile.head()

In [ ]:
# ── §9.3 Religious Sub-Lexicon Heatmap (Religion personas only) ──────────────
religion_personas = sorted([p for p in profile.index 
                            if profile.loc[p,'domain'] == 'religion'])
rel_cols = ['r_generic_d','r_islamic_d','r_christian_d',
            'r_jewish_d','r_secular_d','r_moral_src_d']

hm_data = profile.loc[religion_personas, rel_cols]

fig, ax = plt.subplots(figsize=(11, 8))
sns.heatmap(hm_data, ax=ax, cmap='YlOrRd', linewidths=0.4, linecolor='white',
            annot=True, fmt='.2f',
            xticklabels=[c.replace('_d','').replace('r_','R_') for c in rel_cols],
            yticklabels=hm_data.index,
            cbar_kws={'label': 'Density per 100 words'})
ax.set_title('Religious Sub-Lexicon Density per Religion Persona\n(absolute values)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(CACHE_DIR / 'fig_religious_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"✓ Saved {CACHE_DIR / 'fig_religious_heatmap.png'}")

In [ ]:
# ── §9.4 Comprehensive Marker Heatmap (All Personas, Z-normalized) ──────────
KEY_MARKERS = [
    'r_generic_d','r_islamic_d','r_christian_d','r_jewish_d','r_secular_d',
    'h_hedges_d','h_boosters_d','h_attitude_d','h_engagement_d','h_self_mention_d',
    'h_transitions_d','h_frame_d','h_code_glosses_d','h_evidentials_d',
    'mod_deontic_d','mod_epistemic_d','mod_dynamic_d',
    'c_additive_d','c_adversative_d','c_concessive_d','c_causal_d',
    'rv_research_d','rv_cognition_d','rv_discourse_d',
    'stance_committed','stance_conditional','stance_refusal',
    'opener_document_style','opener_prose_direct','opener_instructional',
    'ai_disc_any', 'word_count',
]
KEY_MARKERS = [m for m in KEY_MARKERS if m in profile.columns]

hm_data = profile[KEY_MARKERS].copy()
hm_norm = (hm_data - hm_data.mean()) / (hm_data.std() + 1e-9)

# Sort: domain → condition
domain_order = {'religion':0,'regional':1,'gender':2,'neutral':3}
cond_order = {'explicit':0,'implicit':1,'neutral':2}
hm_norm['_d'] = [domain_order.get(profile.loc[p,'domain'], 9) for p in hm_norm.index]
hm_norm['_c'] = [cond_order.get(profile.loc[p,'condition'], 9) for p in hm_norm.index]
hm_norm = hm_norm.sort_values(['_d','_c']).drop(columns=['_d','_c'])

fig, ax = plt.subplots(figsize=(18, 18))
sns.heatmap(hm_norm, ax=ax, cmap='RdBu_r', center=0,
            linewidths=0.2, linecolor='#f4f4f4',
            cbar_kws={'label':'Z-score (per marker)', 'shrink':0.5},
            xticklabels=[m.replace('_d','').replace('_',' ') for m in hm_norm.columns],
            yticklabels=hm_norm.index)
ax.set_title('Linguistic Marker Profile per Persona — v3 (Z-normalized)\n'
             'Religious sub-lexica + Hyland metadiscourse + Halliday connectives + reporting verbs',
             fontsize=12, fontweight='bold', pad=10)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=8)
plt.tight_layout()
plt.savefig(CACHE_DIR / 'fig_master_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"✓ Saved {CACHE_DIR / 'fig_master_heatmap.png'}")

## §10 — Statistical Tests
- **Kruskal-Wallis** untuk variasi antar persona (omnibus test per marker)
- **Mann-Whitney U** untuk pairwise explicit-vs-implicit per domain  
- **Bonferroni correction** untuk multiple comparisons
- **Rank-biserial r** sebagai effect size

In [ ]:
# ── §10.1 Mann-Whitney: Explicit vs Implicit per Domain ──────────────────────
def mw_effect_r(x, y):
    x = np.array(x); y = np.array(y)
    x = x[~np.isnan(x)]; y = y[~np.isnan(y)]
    if len(x) < 5 or len(y) < 5: return np.nan, np.nan, np.nan
    u, p = mannwhitneyu(x, y, alternative='two-sided')
    r = 1 - (2*u)/(len(x)*len(y))
    return u, p, r

def label_effect(r):
    r = abs(r) if not np.isnan(r) else 0
    if r >= 0.5: return 'Very Large'
    if r >= 0.3: return 'Large'
    if r >= 0.1: return 'Medium'
    return 'Small'

TEST_MARKERS = [c for c in KEY_MARKERS if c in all_markers.columns 
                and all_markers[c].dtype in ('float64','int64','int32','float32')]

results = []
for marker in TEST_MARKERS:
    for domain in ['religion','regional','gender']:
        ex = all_markers[(all_markers['domain']==domain) & (all_markers['condition']=='explicit')][marker].dropna()
        im = all_markers[(all_markers['domain']==domain) & (all_markers['condition']=='implicit')][marker].dropna()
        u, p, r = mw_effect_r(ex, im)
        results.append({
            'marker':marker, 'domain':domain,
            'mean_exp':ex.mean(), 'mean_imp':im.mean(),
            'n_exp':len(ex), 'n_imp':len(im),
            'U':u, 'p_raw':p, 'r':r,
            'effect':label_effect(r),
        })

stats_df = pd.DataFrame(results)
mask = ~stats_df['p_raw'].isna()
_, p_corr, _, _ = multipletests(stats_df.loc[mask,'p_raw'], method='bonferroni')
stats_df.loc[mask,'p_corrected'] = p_corr
stats_df['significant'] = stats_df['p_corrected'] < 0.05

sig = stats_df[stats_df['significant']].sort_values('r', key=abs, ascending=False)
print(f"Significant (Bonferroni p<0.05): {len(sig)}/{len(stats_df)}\n")
print("Top 20 effect sizes:")
print(sig[['marker','domain','mean_exp','mean_imp','r','effect','p_corrected']].head(20).to_string(index=False))

In [ ]:
# ── §10.2 Kruskal-Wallis Omnibus per Marker (all 49 personas) ────────────────
kw_results = []
personas = all_markers['persona'].unique()

for marker in TEST_MARKERS:
    groups = []
    for p in personas:
        v = all_markers[all_markers['persona']==p][marker].dropna().values
        if len(v) >= 5: groups.append(v)
    if len(groups) < 3: continue
    try:
        h, p_val = kruskal(*groups)
        k = len(groups); N = sum(len(g) for g in groups)
        eta_sq = max((h - k + 1) / (N - k), 0)
        kw_results.append({'marker':marker, 'H':h, 'p':p_val, 
                            'eta_sq':eta_sq, 'k':k, 'N':N})
    except: pass

kw_df = pd.DataFrame(kw_results).sort_values('H', ascending=False)
kw_df['sig'] = kw_df['p'] < 0.001  # strict threshold

print(f"Kruskal-Wallis results ({len(kw_df)} markers tested):\n")
print(f"  Significant @ p<0.001 : {kw_df['sig'].sum()}/{len(kw_df)}")
print()
print("Top 15 by H statistic:")
print(kw_df.head(15)[['marker','H','eta_sq','p']].to_string(index=False))

## §11 — Synthesis & Thesis-Ready Outputs

In [ ]:
# ── §11.1 Religion Persona Comparison (key takeaway) ─────────────────────────
print("=" * 76)
print("RELIGION PERSONA — RELIGIOUS SUB-LEXICON SIGNATURE")
print("=" * 76)
print()
print(f"{'Persona':<32} {'R_gen':>7} {'R_isl':>7} {'R_chr':>7} {'R_jew':>7} {'R_sec':>7}")
print("─" * 76)
for p in sorted(religion_personas):
    row = profile.loc[p, ['r_generic_d','r_islamic_d','r_christian_d',
                          'r_jewish_d','r_secular_d']]
    print(f"{p:<32} {row.iloc[0]:>7.3f} {row.iloc[1]:>7.3f} {row.iloc[2]:>7.3f} {row.iloc[3]:>7.3f} {row.iloc[4]:>7.3f}")
print()

# Sanity check: each religion explicit should dominate its OWN sublex
print("SANITY CHECK — Does each religion persona dominate its own sub-lexicon?")
expected = {
    'islam_explicit'    : 'r_islamic_d',
    'christian_explicit': 'r_christian_d',
    'jewish_explicit'   : 'r_jewish_d',
    'atheist_explicit'  : 'r_secular_d',
}
for persona, expected_col in expected.items():
    if persona not in profile.index: continue
    row = profile.loc[persona, rel_cols]
    actual_max = row.idxmax()
    match = "✓" if actual_max == expected_col else "✗"
    print(f"  {match} {persona:<25} expected:{expected_col:<15} got:{actual_max:<15}")

In [ ]:
# ── §11.2 Master Ranking: Persona Distinctiveness ────────────────────────────
# Distance from neutral in Z-normalized marker space
if 'neutral' in profile.index:
    profile_num = profile[KEY_MARKERS].values
    profile_norm = (profile_num - profile_num.mean(axis=0)) / (profile_num.std(axis=0) + 1e-9)
    neutral_idx = list(profile.index).index('neutral')
    distances = np.linalg.norm(profile_norm - profile_norm[neutral_idx], axis=1)
    profile['dist_from_neutral'] = distances
    
    ranking = profile[['dist_from_neutral','domain','condition','n']].sort_values(
        'dist_from_neutral', ascending=False)
    
    print("=" * 76)
    print("TOP 15 MOST LINGUISTICALLY DISTINCTIVE PERSONAS")
    print("=" * 76)
    print(ranking.head(15).to_string())
    print()
    print("BOTTOM 10 (closest to neutral):")
    print(ranking.tail(10).to_string())

# Export full results for thesis appendix
out_path = CACHE_DIR / 'thesis_marker_profile_v3.csv'
profile.to_csv(out_path)
print(f"\n✓ Exported → {out_path}")

In [ ]:
# ── §11.3 Final Summary Print ────────────────────────────────────────────────
print("=" * 76)
print("LINGUISTIC MARKER v3 — COMPREHENSIVE SUMMARY")
print("=" * 76)

print(f"\n[Dataset]")
print(f"  Total responses          : {len(df):,}")
print(f"  Personas                 : {df['persona'].nunique()}")
print(f"  None rate                : {df['is_none'].mean():.1%}")

print(f"\n[Marker Coverage v3]")
print(f"  Religious sub-lexica     : 6 categories ({len(rel_cols)} markers)")
print(f"  Hyland metadiscourse     : {len([c for c in profile.columns if c.startswith('h_')])} categories")
print(f"  Modal verbs (Palmer)     : {len([c for c in profile.columns if c.startswith('mod_')])} categories")
print(f"  Halliday connectives     : {len([c for c in profile.columns if c.startswith('c_')])} categories")
print(f"  Reporting verbs (Hyland) : {len([c for c in profile.columns if c.startswith('rv_')])} categories")
print(f"  Stance + opener types    : {len([c for c in profile.columns if c.startswith('stance_') or c.startswith('opener_')])} categories")
print(f"  AI disclaimer            : 1 (binary)")

print(f"\n[Religious Lexicon Coverage Gain (vs v2 MFD-sanctity only)]")
v2_count = (rel_df['r_generic_d'] > 0).sum()  # approximate v2 coverage
v3_count = (rel_df['r_total_d'] > 0).sum() if 'r_total_d' in rel_df.columns else 0
print(f"  v2 (sanctity-only)       : ~933 responses with religious lexicon")
print(f"  v3 (6 sub-lexica)        : {((rel_df.iloc[:,:6] > 0).any(axis=1)).sum():,} responses ({((rel_df.iloc[:,:6] > 0).any(axis=1)).mean()*100:.1f}%)")

print(f"\n[Statistical Significance]")
print(f"  Mann-Whitney (exp vs imp): {len(sig)}/{len(stats_df)} significant @ Bonferroni p<0.05")
print(f"  Kruskal-Wallis (49 pers.): {kw_df['sig'].sum()}/{len(kw_df)} significant @ p<0.001")

print(f"\n[Files exported]")
for f in CACHE_DIR.iterdir():
    if f.suffix in ('.png','.csv','.pkl'):
        size_kb = f.stat().st_size / 1024
        print(f"  {f.name:<40} {size_kb:>8.1f} KB")

print("\n" + "=" * 76)
print("✓ ANALYSIS COMPLETE — Ready for thesis integration")
print("=" * 76)